# Módulo F2 · SQL para análisis de datos — Práctica guiada

**Caso:** eres analista en *TiendaSur*. Dirección quiere responder estas preguntas con datos:

1. ¿Qué categorías venden más?
2. ¿Qué productos son los más rentables por categoría?
3. ¿Cómo evolucionan las ventas mes a mes y cuál es el **acumulado**?

Todos los datos son sintéticos y se crean en la celda siguiente. La base de datos es **SQLite** (un archivo local) y la consultamos con `pandas.read_sql`.

> **Ejecuta las celdas en orden** (el kernel no recuerda entre sesiones).


In [ ]:
import sqlite3
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# --- Base de datos sintética de TiendaSur -------------------------------------
clientes = pd.DataFrame({
    "cliente_id": range(1, 41),
    "nombre": [f"Cliente {i:02d}" for i in range(1, 41)],
    "ciudad": rng.choice(["Madrid", "Barcelona", "Valencia", "Sevilla"], 40, p=[.35, .3, .2, .15]),
})

categorias = ["Bebidas", "Alimentación", "Limpieza", "Electrónica"]
productos = pd.DataFrame({
    "producto_id": range(1, 25),
    "nombre": [f"Producto {i:02d}" for i in range(1, 25)],
    "categoria": np.tile(categorias, 6),
    "precio": np.round(rng.uniform(1, 60, 24), 2),
})
productos.loc[3, "precio"] = np.nan   # un nulo para practicar COALESCE

# Pedidos: 400 pedidos repartidos en 2024
n_pedidos = 400
pedidos = pd.DataFrame({
    "pedido_id": range(1, n_pedidos + 1),
    "cliente_id": rng.integers(1, 41, n_pedidos),
    "fecha": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 364, n_pedidos), unit="D"),
})

# Líneas de pedido: entre 1 y 4 productos por pedido
lineas = []
for _, p in pedidos.iterrows():
    n_l = rng.integers(1, 5)
    elegidos = rng.choice(productos["producto_id"], size=min(n_l, 24), replace=False)
    for pid in elegidos:
        unid = rng.integers(1, 8)
        precio = float(productos.loc[productos.producto_id == pid, "precio"].iloc[0])
        if np.isnan(precio):
            precio = 10.0   # el ticket nunca es nulo: precio medio como fallback
        lineas.append({"pedido_id": int(p["pedido_id"]), "producto_id": int(pid),
                       "unidades": int(unid), "precio_unitario": precio})
lineas = pd.DataFrame(lineas)
lineas["importe"] = lineas["unidades"] * lineas["precio_unitario"]

# Guardamos en SQLite
conn = sqlite3.connect(":memory:")
for nombre, df in [("clientes", clientes), ("productos", productos), ("pedidos", pedidos), ("lineas", lineas)]:
    df.to_sql(nombre, conn, index=False)

print(f"clientes: {len(clientes)}  productos: {len(productos)}  pedidos: {len(pedidos)}  líneas: {len(lineas)}")
conn

## 2.1–2.2 · SELECT básico: explorar las tablas

Empezemos por la tabla `productos`. ¿Hay nulos? ¿Qué rango de precios?

In [ ]:
# Primera consulta: mira solo 5 filas para no volcar toda la tabla
pd.read_sql("""
SELECT producto_id, nombre, categoria, precio
FROM productos
ORDER BY precio DESC
LIMIT 5;
""", conn)

In [ ]:
# ¿Dónde está el precio nulo? Recuerda: con NULL no se usa = sino IS NULL
pd.read_sql("""
SELECT *
FROM productos
WHERE precio IS NULL;
""", conn)

In [ ]:
# COALESCE para que el informe no muestre huecos
pd.read_sql("""
SELECT nombre, categoria,
       COALESCE(precio, 0) AS precio_seguro
FROM productos
WHERE precio IS NULL OR categoria = 'Bebidas';
""", conn)

## 2.3 · Fechas: agrupar por mes

SQLite no tiene `date_trunc`, usamos `strftime('%Y-%m', fecha)`. Esto devuelve '2024-01', '2024-02'… y lo usaremos como clave de mes durante todo el notebook.

In [ ]:
pd.read_sql("""
SELECT fecha,
       strftime('%Y-%m', fecha) AS mes
FROM pedidos
ORDER BY fecha
LIMIT 8;
""", conn)

## 2.4 · JOIN: unir pedidos con líneas y clientes

In [ ]:
# Pedido 1: ¿quién es el cliente?
pd.read_sql("""
SELECT p.pedido_id, p.fecha, c.nombre, c.ciudad
FROM pedidos p
JOIN clientes c ON c.cliente_id = p.cliente_id
ORDER BY p.fecha
LIMIT 8;
""", conn)

In [ ]:
# ⚠️ Comprobación de seguridad: el join no debe "multiplicar" filas
antes = pd.read_sql("SELECT COUNT(*) AS n FROM pedidos", conn)["n"].iloc[0]
despues = pd.read_sql("""
SELECT COUNT(*) AS n
FROM pedidos p
JOIN lineas l ON l.pedido_id = p.pedido_id;
""", conn)["n"].iloc[0]
print(f"pedidos: {antes}   con líneas: {despues}  (crece porque cada un pedido tiene varias líneas)")

## 2.5 · GROUP BY: ventas por categoría

In [ ]:
pd.read_sql("""
SELECT pr.categoria,
       COUNT(*) AS n_lineas,
       SUM(l.unidades) AS unidades,
       ROUND(SUM(l.importe), 2) AS ingresos
FROM lineas l
JOIN productos pr ON pr.producto_id = l.producto_id
GROUP BY pr.categoria
ORDER BY ingresos DESC;
""", conn)

In [ ]:
# HAVING: solo categorías con más de 200 unidades
pd.read_sql("""
SELECT pr.categoria, SUM(l.unidades) AS unidades
FROM lineas l
JOIN productos pr ON pr.producto_id = l.producto_id
GROUP BY pr.categoria
HAVING SUM(l.unidades) > 200;
""", conn)

## 2.6 · CTE: el ticket medio por cliente, solo de clientes de Madrid

Leemos la consulta «en capas»: primero calculamos el gasto total por cliente, luego filtramos.

In [ ]:
pd.read_sql("""
WITH gasto_cliente AS (
  SELECT c.cliente_id, c.nombre, c.ciudad,
         ROUND(SUM(l.importe), 2) AS gasto_total
  FROM lineas l
  JOIN pedidos p ON p.pedido_id = l.pedido_id
  JOIN clientes c ON c.cliente_id = p.cliente_id
  GROUP BY c.cliente_id
)
SELECT *
FROM gasto_cliente
WHERE ciudad = 'Madrid'
ORDER BY gasto_total DESC
LIMIT 10;
""", conn)

## 2.7 · Funciones de ventana: top-3 por categoría, acumulado y comparación mes a mes

Aquí está lo que más se echa en falta si solo sabes `GROUP BY`.

In [ ]:
# Top-3 productos por categoría (por unidades vendidas)
pd.read_sql("""
WITH vendido AS (
  SELECT pr.categoria, pr.nombre, SUM(l.unidades) AS unidades
  FROM lineas l
  JOIN productos pr ON pr.producto_id = l.producto_id
  GROUP BY pr.producto_id
),
con_rank AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY unidades DESC) AS rn
  FROM vendido
)
SELECT categoria, nombre, unidades, rn
FROM con_rank
WHERE rn <= 3
ORDER BY categoria, rn;
""", conn)

In [ ]:
# Total acumulado por mes (cuánto hemos vendido hasta cada mes)
pd.read_sql("""
WITH meses AS (
  SELECT strftime('%Y-%m', p.fecha) AS mes, l.importe
  FROM lineas l
  JOIN pedidos p ON p.pedido_id = l.pedido_id
),
mes_total AS (
  SELECT mes, ROUND(SUM(importe), 2) AS mes_importe
  FROM meses
  GROUP BY mes
)
SELECT mes, mes_importe,
       ROW_NUMBER() OVER (ORDER BY mes) AS mes_num,
       ROUND(SUM(mes_importe) OVER (ORDER BY mes), 2) AS acumulado
FROM mes_total
ORDER BY mes;
""", conn)

In [ ]:
# Diferencia con el mes anterior (LAG)
pd.read_sql("""
WITH mes_total AS (
  SELECT strftime('%Y-%m', p.fecha) AS mes, ROUND(SUM(l.importe), 2) AS mes_importe
  FROM lineas l
  JOIN pedidos p ON p.pedido_id = l.pedido_id
  GROUP BY mes
)
SELECT mes, mes_importe,
       LAG(mes_importe) OVER (ORDER BY mes) AS mes_anterior,
       ROUND(mes_importe - LAG(mes_importe) OVER (ORDER BY mes), 2) AS diferencia
FROM mes_total
ORDER BY mes;
""", conn)

## Resumen del notebook

- Filtra y agrega en **SQL**, pasa el resultado a `pandas` para visualizar.
- Los JOIN multiplican filas: comprueba siempre el `COUNT(*)` después.
- **CTE** para leer consultas en capas; **ventana** para rankings, cuotas y acumulados.

---

## Ejercicios (5) — hazlo para la entrega

Para cada uno: escribe la consulta, muéstrala ejecutada y redacta **una frase de conclusión para Dirección**.

**E1.** Total de ingresos y número de pedidos por ciudad del cliente (`clientes.ciudad`). Ordena por ingresos descendentes.

**E2.** Los 5 clientes que más han gastado en total (nombre + gasto).

**E3.** Ticket medio por mes (importe total del mes / número de pedidos del mes). *Pista:* junta dos CTE o usa COUNT(DISTINCT pedido_id).

**E4.** La *cuota* de ventas de cada producto dentro de su categoría: `unidades / SUM(unidades) OVER (PARTITION BY categoria)`, en tanto por ciento.

**E5.** Encuentra los pedidos con **más líneas** que el pedido 100 (usa una subconsulta o CTE con `HAVING`).
